# Part 5 — Analytics library walkthrough (IV, ATM, term structure, skew, greeks)

This notebook walks through the deterministic computations implemented in:

- `src/voli/analytics/iv_metrics.py`
- `src/voli/analytics/skew.py`
- `src/voli/analytics/greeks.py`

**Goal:** the agent never “wings it” — it calls tools for data, then calls this library for metrics.

## 0) Setup

Run from your repo root. This cell makes `src/` importable so `import voli...` works even if you haven't installed the package in editable mode.

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
src = repo_root / "src"
if src.exists():
    sys.path.insert(0, str(src))

# If you prefer, you can instead do:
#   pip install -e .

## 1) Import the analytics modules

We'll use `inspect.getsource(...)` a lot to show the exact code you're running locally.

In [2]:
import inspect

from voli.analytics import greeks, iv_metrics, skew

## 2) Core conventions (v1)

- **ATM selection:** nearest strike to spot, deterministic tie-break (lower strike on exact tie).
- **Mid price:** prefer `(bid+ask)/2` if both valid; otherwise deterministic fallbacks (last → bid-only → ask-only → None).
- **Spread filter:** relative spread = `(ask-bid)/mid`; “too wide” if above threshold.
- **Missing data:** return `None` values plus **flags** the caller can turn into `PARTIAL_DATA`.

## 3) `MetricResult`: the “value + flags” container

Most functions return `MetricResult[T]` so callers can keep computations deterministic while still surfacing data quality.

In [3]:
print(inspect.getsource(iv_metrics.MetricResult))

@dataclass(frozen=True)
class MetricResult(Generic[T]):
    """A deterministic value plus flags for partial/missing data."""

    value: T | None
    flags: tuple[str, ...] = ()

    def with_flag(self, flag: str) -> MetricResult[T]:
        if flag in self.flags:
            return self
        return MetricResult(self.value, self.flags + (flag,))



## 4) Mid price and spread logic

### Mid price rules (in order)
1. If bid+ask present, non-negative, and `ask >= bid`: `mid=(bid+ask)/2`.
2. Else if `last` present: use `last` (flag).
3. Else if only `bid` present: use bid (flag).
4. Else if only `ask` present: use ask (flag).
5. Else: `None` + flags.

In [4]:
print(inspect.getsource(iv_metrics.mid_price))

def mid_price(
    bid: float | None, ask: float | None, last: float | None = None
) -> MetricResult[float]:
    """Compute a deterministic mid price.

    Rules (in order):
    1) If bid and ask are present, non-negative, and ask >= bid: mid = (bid + ask)/2.
    2) Else if last is present and non-negative: use last.
    3) Else if only bid is present and non-negative: use bid.
    4) Else if only ask is present and non-negative: use ask.
    5) Else: None.

    Flags are added when we fall back from the ideal case.
    """

    flags: list[str] = []

    def _ok(x: float | None) -> bool:
        return x is not None and x >= 0

    if _ok(bid) and _ok(ask) and ask >= bid:  # type: ignore[operator]
        return MetricResult((bid + ask) / 2, ())

    flags.append("MID_NOT_FROM_BIDASK")

    if _ok(last):
        return MetricResult(last, tuple(flags) + ("MID_FROM_LAST",))

    if _ok(bid) and ask is None:
        return MetricResult(bid, tuple(flags) + ("MID_FROM_BID_ONLY",))

    if 

### Spread logic

- `relative_spread(bid, ask, mid)` returns `None` if the spread can't be computed.
- `is_spread_too_wide(...)` compares that relative spread to a threshold (default used in tests: `0.20`).

In [5]:
print(inspect.getsource(iv_metrics.relative_spread))
print(inspect.getsource(iv_metrics.is_spread_too_wide))

def relative_spread(bid: float | None, ask: float | None) -> MetricResult[float]:
    """Relative spread = (ask - bid) / mid, where mid=(bid+ask)/2.

    Returns None if bid/ask missing or invalid.
    """

    if bid is None:
        return MetricResult(None, ("MISSING_BID", "SPREAD_MISSING"))
    if ask is None:
        return MetricResult(None, ("MISSING_ASK", "SPREAD_MISSING"))
    if bid < 0 or ask < 0:
        return MetricResult(None, ("NEGATIVE_QUOTE", "SPREAD_MISSING"))
    if ask < bid:
        return MetricResult(None, ("ASK_LT_BID", "SPREAD_MISSING"))

    mid = (bid + ask) / 2
    if mid == 0:
        return MetricResult(None, ("MID_ZERO", "SPREAD_MISSING"))

    return MetricResult((ask - bid) / mid, ())

def is_spread_too_wide(
    bid: float | None,
    ask: float | None,
    *,
    max_relative_spread: float = 0.20,
) -> MetricResult[bool]:
    """True if relative spread exceeds threshold.

    If spread can't be computed, returns None + flags.
    """

    rs = relati

#### Quick sanity checks

In [6]:
examples = [
    dict(bid=9.0, ask=11.0, last=None),
    dict(bid=None, ask=11.0, last=10.0),
    dict(bid=9.0, ask=None, last=None),
    dict(bid=None, ask=None, last=None),
]

for ex in examples:
    m = iv_metrics.mid_price(**ex)
    rs = iv_metrics.relative_spread(ex["bid"], ex["ask"])
    too_wide = iv_metrics.is_spread_too_wide(ex["bid"], ex["ask"], max_relative_spread=0.20).value
    print(ex, "=> mid:", m, "rel_spread:", rs, "too_wide:", too_wide)

{'bid': 9.0, 'ask': 11.0, 'last': None} => mid: MetricResult(value=10.0, flags=()) rel_spread: MetricResult(value=0.2, flags=()) too_wide: False
{'bid': None, 'ask': 11.0, 'last': 10.0} => mid: MetricResult(value=10.0, flags=('MID_NOT_FROM_BIDASK', 'MID_FROM_LAST')) rel_spread: MetricResult(value=None, flags=('MISSING_BID', 'SPREAD_MISSING')) too_wide: None
{'bid': 9.0, 'ask': None, 'last': None} => mid: MetricResult(value=9.0, flags=('MID_NOT_FROM_BIDASK', 'MID_FROM_BID_ONLY')) rel_spread: MetricResult(value=None, flags=('MISSING_ASK', 'SPREAD_MISSING')) too_wide: None
{'bid': None, 'ask': None, 'last': None} => mid: MetricResult(value=None, flags=('MID_NOT_FROM_BIDASK', 'MISSING_BID', 'MISSING_ASK', 'MISSING_LAST', 'MID_MISSING')) rel_spread: MetricResult(value=None, flags=('MISSING_BID', 'SPREAD_MISSING')) too_wide: None


## 5) ATM strike selection

ATM is **spot-based** nearest strike. If spot is exactly halfway between two strikes, we pick the **lower** strike to be deterministic.

In [7]:
print(inspect.getsource(iv_metrics.select_atm_strike))

def select_atm_strike(spot: float, strikes: Iterable[float]) -> MetricResult[float]:
    """Select nearest strike to spot (deterministic tie-break).

    Tie-break: if two strikes are equally distant, choose the *lower* strike.

    Returns None if strikes is empty.
    """

    strikes_list = list(strikes)
    if not strikes_list:
        return MetricResult(None, ("NO_STRIKES",))

    # Sort ensures deterministic tie-break.
    strikes_list.sort()

    best = strikes_list[0]
    best_dist = abs(best - spot)

    for k in strikes_list[1:]:
        dist = abs(k - spot)
        if dist < best_dist:
            best = k
            best_dist = dist
        elif dist == best_dist and k < best:
            best = k

    return MetricResult(best, ())



## 6) Build a synthetic options grid (offline demo)

These simple dataclasses mimic your internal models:
- contracts: `expiry`, `strike`, `right`, `option_symbol`
- greeks: `iv`, `delta`, `gamma`, `theta`, `vega`

Your real pipeline will pass `OptionContract` and `OptionGreeks` objects.

In [8]:
from dataclasses import dataclass
from datetime import date


@dataclass(frozen=True)
class C:
    option_symbol: str
    expiry: date
    strike: float
    right: str  # "call" / "put"


@dataclass(frozen=True)
class G:
    iv: float | None
    delta: float | None = None
    gamma: float | None = None
    theta: float | None = None
    vega: float | None = None


front = date(2026, 1, 17)
next_ = date(2026, 2, 21)

spot = 102.0
strikes = [95, 100, 105, 110]

contracts = []
greeks_by_symbol = {}

# deterministic IV pattern:
# - longer expiry has higher IV
# - puts higher than calls (skew)
for expiry, base in [(front, 0.30), (next_, 0.35)]:
    for k in strikes:
        for right in ["call", "put"]:
            sym = f"O:{expiry.isoformat()}:{right}:{k}"
            contracts.append(C(sym, expiry, float(k), right))
            bump = 0.00
            if expiry == next_:
                bump += 0.05  # term structure
            if right == "put":
                bump += 0.03  # put skew
            # mild smile around 100
            bump += 0.002 * abs(k - 100)

            iv = base + bump
            greeks_by_symbol[sym] = G(iv=iv, delta=None, gamma=None, theta=None, vega=None)

len(contracts), list(greeks_by_symbol.items())[:2]

(16,
 [('O:2026-01-17:call:95',
   G(iv=0.31, delta=None, gamma=None, theta=None, vega=None)),
  ('O:2026-01-17:put:95',
   G(iv=0.33999999999999997, delta=None, gamma=None, theta=None, vega=None))])

## 7) Term structure: front vs next expiry at same strike

`atm_iv_term_structure(...)`:
- finds **two nearest expiries** for a given right
- chooses ATM strike using the **front expiry** strike grid
- compares IV for that **same strike** for front vs next expiry

In [9]:
print(inspect.getsource(iv_metrics.atm_iv_term_structure))

def atm_iv_term_structure(
    *,
    spot: float,
    contracts: Sequence[object],
    greeks_by_symbol: Mapping[str, object],
    right: str = "call",
) -> TermStructureResult:
    """Compare ATM IV for the two nearest expiries using the *same strike*.

    Strategy:
    - Find the two earliest expiries available for the given right.
    - Choose ATM strike using the *front* expiry strike grid.
    - Look up IV for that strike for front and next expiry.

    Assumes:
    - contract has attrs: expiry, strike, right, option_symbol (or symbol)
    - greeks object has attr: iv
    """

    right_n = normalize_right(right)

    # Collect expiries for this right.
    expiries = sorted(
        {_as_date(c.expiry) for c in contracts if normalize_right(c.right) == right_n}
    )
    if len(expiries) == 0:
        return TermStructureResult(None, None, None, None, None, ("NO_EXPIRIES",))
    if len(expiries) == 1:
        return TermStructureResult(None, expiries[0], None, None, None, ("ONLY_

In [ ]:
ts_call = iv_metrics.atm_iv_term_structure(
    spot=spot,
    contracts=contracts,
    greeks_by_symbol=greeks_by_symbol,
    right="call",
)
ts_put = iv_metrics.atm_iv_term_structure(
    spot=spot,
    contracts=contracts,
    greeks_by_symbol=greeks_by_symbol,
    right="put",
)

ts_call, ts_put

(TermStructureResult(atm_strike=100.0, front_expiry=datetime.date(2026, 1, 17), next_expiry=datetime.date(2026, 2, 21), front_iv=0.3, next_iv=0.39999999999999997, flags=()),
 TermStructureResult(atm_strike=100.0, front_expiry=datetime.date(2026, 1, 17), next_expiry=datetime.date(2026, 2, 21), front_iv=0.32999999999999996, next_iv=0.43, flags=()))

### Missing-data behavior demo

Remove an IV for the next expiry ATM strike and re-run to see `MISSING_IV` flags.

In [ ]:
# Remove IV for the "next" call at the ATM strike (expected strike ~ 100 or 105 based on spot=102)
atm_strike = ts_call.atm_strike
target_sym = f"O:{next_.isoformat()}:call:{int(atm_strike)}"
greeks_by_symbol_missing = dict(greeks_by_symbol)
greeks_by_symbol_missing[target_sym] = G(iv=None)

ts_missing = iv_metrics.atm_iv_term_structure(
    spot=spot,
    contracts=contracts,
    greeks_by_symbol=greeks_by_symbol_missing,
    right="call",
)
ts_missing

TermStructureResult(atm_strike=100.0, front_expiry=datetime.date(2026, 1, 17), next_expiry=datetime.date(2026, 2, 21), front_iv=0.3, next_iv=None, flags=('MISSING_IV',))

## 8) Skew: strike→IV pairs and slope

`strike_iv_pairs(...)` returns the IV curve for a given expiry/right, sorted by strike.
`skew_slope(...)` computes a simple OLS slope for `iv ~ strike`.

Interpretation:
- **Negative slope for puts** is common (higher IV at lower strikes).
- Calls may be flatter or slightly positive depending on market and underlying.

In [ ]:
print(inspect.getsource(skew.strike_iv_pairs))
print(inspect.getsource(skew.skew_slope))

def strike_iv_pairs(
    *,
    contracts: Sequence[object],
    greeks_by_symbol: Mapping[str, object],
    expiry: date | datetime | str,
    right: str,
) -> MetricResult[SkewCurve]:
    """Return (strike, iv) pairs for a single expiry/right.

    Assumes contract has attrs: expiry, strike, right, option_symbol (or symbol).
    Assumes greeks has attr: iv.

    Any contract whose IV is missing is skipped and flagged.
    """

    exp = _as_date(expiry)
    r = normalize_right(right)

    flags: list[str] = []
    pairs: list[tuple[float, float]] = []

    # Deterministic ordering: by strike, then option symbol.
    def _key(c: object) -> tuple[float, str]:
        sym = getattr(c, "option_symbol", None) or getattr(c, "symbol", None) or ""
        return (float(c.strike), sym)

    filtered = [c for c in contracts if _as_date(c.expiry) == exp and normalize_right(c.right) == r]

    if not filtered:
        return MetricResult(None, ("NO_CONTRACTS_FOR_EXPIRY_RIGHT",))

    for c in so

In [ ]:
pairs_put_front = skew.strike_iv_pairs(
    contracts=contracts,
    greeks_by_symbol=greeks_by_symbol,
    expiry=front,
    right="put",
).value

pairs_call_front = skew.strike_iv_pairs(
    contracts=contracts,
    greeks_by_symbol=greeks_by_symbol,
    expiry=front,
    right="call",
).value

pairs_put_front, pairs_call_front

(SkewCurve(expiry=datetime.date(2026, 1, 17), right='put', pairs=((95.0, 0.33999999999999997), (100.0, 0.32999999999999996), (105.0, 0.33999999999999997), (110.0, 0.35)), flags=()),
 SkewCurve(expiry=datetime.date(2026, 1, 17), right='call', pairs=((95.0, 0.31), (100.0, 0.3), (105.0, 0.31), (110.0, 0.32)), flags=()))

In [14]:
slope_put_front = skew.skew_slope(
    contracts=contracts,
    greeks_by_symbol=greeks_by_symbol,
    expiry=front,
    right="put",
).value

slope_call_front = skew.skew_slope(
    contracts=contracts,
    greeks_by_symbol=greeks_by_symbol,
    expiry=front,
    right="call",
).value

slope_put_front, slope_call_front

(0.0008000000000000007, 0.0008000000000000007)

## 9) Greeks helpers

This module is intentionally small:
- lookup an option's greeks deterministically via `option_symbol`
- select the ATM contract for a given expiry/right and return its greek snapshot

In [15]:
print(inspect.getsource(greeks.atm_greeks_for_expiry))

def atm_greeks_for_expiry(
    *,
    spot: float,
    contracts: Sequence[object],
    greeks_by_symbol: Mapping[str, object],
    expiry: date | datetime | str,
    right: str,
) -> MetricResult[GreeksSnapshot]:
    """Select the ATM contract for expiry/right and return its greeks.

    - ATM strike chosen from strike grid for that expiry/right.
    - If the greeks object is missing or individual fields are missing, we return None values
      plus flags.

    Assumes contract has attrs: expiry, strike, right, option_symbol (or symbol).
    """

    exp = _as_date(expiry)
    r = normalize_right(right)

    strikes = [
        float(c.strike)
        for c in contracts
        if _as_date(c.expiry) == exp and normalize_right(c.right) == r
    ]
    atm = select_atm_strike(spot, strikes)
    if atm.value is None:
        return MetricResult(None, atm.flags + ("ATM_STRIKE_MISSING",))

    c_res = select_contract_by_strike(contracts, strike=atm.value, right=r, expiry=exp)
    if c_res.v

For this synthetic demo we didn't populate delta/gamma/theta/vega, so you'll see flags for missing fields if you request them.

In your real pipeline, `greeks_by_symbol` will come from tools and usually has these values.

In [16]:
atm_g = greeks.atm_greeks_for_expiry(
    spot=spot,
    contracts=contracts,
    greeks_by_symbol=greeks_by_symbol,
    expiry=front,
    right="call",
)
atm_g

MetricResult(value=GreeksSnapshot(option_symbol='O:2026-01-17:call:100', expiry=datetime.date(2026, 1, 17), strike=100.0, right='call', iv=0.3, delta=None, gamma=None, theta=None, vega=None, flags=('MISSING_DELTA', 'MISSING_GAMMA', 'MISSING_THETA', 'MISSING_VEGA')), flags=('MISSING_DELTA', 'MISSING_GAMMA', 'MISSING_THETA', 'MISSING_VEGA'))

## 10) Where definitions live

The human-readable, explicit definitions are in:

- `docs/metrics_definitions.md`

Open it to confirm tie-break rules and missing-data flags.

## 11) Optional: run the unit tests

From your terminal (repo root):

```bash
poetry run python -m pytest -q
```